## 전이학습으로 돌려보기

In [1]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm  # TQDM import

# Custom dataset
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.data = []
        
        for label in range(len(self.classes)):
            class_folder = os.path.join(root_dir, self.classes[label])
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                self.data.append((img_path, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')  # 이미지 RGB로 변환
        if self.transform:
            image = self.transform(image)
        return image, label

# 경로 및 배치 크기 설정
data_dir = "."
batch_size = 32

# 데이터 증강 포함한 이미지 전처리
transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),  # 랜덤 가로 뒤집기
    T.RandomRotation(10),  # 랜덤 회전
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # 색상 변형
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 학습 및 검증 데이터셋 생성
train_dataset = CustomDataset(os.path.join(data_dir, 'train'), transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = CustomDataset(os.path.join(data_dir, 'valid'), transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

# 사전 학습된 resnext50_32x4d 모델 사용
model = models.resnext50_32x4d(pretrained=True)

# 전이 학습을 위해 일부 가중치 고정 (freeze)
for param in model.parameters():
    param.requires_grad = False

# 출력 레이어를 분류하려는 클래스 수에 맞게 수정
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.5),  # Dropout 추가
    torch.nn.Linear(model.fc.in_features, len(train_dataset.classes))
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 손실 함수 및 옵티마이저 설정
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.fc.parameters(), lr=0.0001, momentum=0.9)

# 학습률 스케줄러 추가 (학습률 점진적 감소)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# 학습 과정 (TQDM으로 진행 상황 시각화)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # TQDM으로 학습 진행 표시
    train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, labels in train_loader_iter:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        train_loader_iter.set_postfix(loss=running_loss / len(train_loader))
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")
    
    # 학습률 스케줄러 업데이트
    scheduler.step()

# 모델 저장
torch.save(model.state_dict(), 'resnext_model.pth')

# 검증 평가
model.eval()
correct = 0
total = 0
with torch.no_grad():
    valid_loader_iter = tqdm(valid_loader, desc="Validating")  # 검증도 TQDM으로 진행 상황 시각화
    for images, labels in valid_loader_iter:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Validation Accuracy: {accuracy}%')

C:\Users\iyoun\AppData\Local\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\iyoun\AppData\Local\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/10: 100%|█████████████████████████████████████████████████████████| 322/322 [02:15<00:00,  2.38it/s, loss=1.41]


Epoch 1/10, Loss: 1.4122263952071623


Epoch 2/10: 100%|█████████████████████████████████████████████████████████| 322/322 [01:38<00:00,  3.28it/s, loss=1.41]


Epoch 2/10, Loss: 1.405086084182218


Epoch 3/10: 100%|█████████████████████████████████████████████████████████| 322/322 [01:38<00:00,  3.26it/s, loss=1.39]


...

Epoch 9/10, Loss: 1.358203569184179


Epoch 10/10: 100%|████████████████████████████████████████████████████████| 322/322 [01:38<00:00,  3.27it/s, loss=1.36]


Epoch 10/10, Loss: 1.3643098880785594


Validating: 100%|██████████████████████████████████████████████████████████████████████| 41/41 [00:16<00:00,  2.48it/s]

Validation Accuracy: 34.26791277258567%


## 전이학습 끈 상태로 전체 레이어 학습 / 학습률 반으로 줄임

In [2]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm  # TQDM import

# Custom dataset
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.data = []
        
        for label in range(len(self.classes)):
            class_folder = os.path.join(root_dir, self.classes[label])
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                self.data.append((img_path, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')  # 이미지 RGB로 변환
        if self.transform:
            image = self.transform(image)
        return image, label

# 경로 및 배치 크기 설정
data_dir = "."
batch_size = 32

# 데이터 증강 포함한 이미지 전처리
transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),  # 랜덤 가로 뒤집기
    T.RandomRotation(10),  # 랜덤 회전
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # 색상 변형
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 학습 및 검증 데이터셋 생성
train_dataset = CustomDataset(os.path.join(data_dir, 'train'), transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = CustomDataset(os.path.join(data_dir, 'valid'), transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

# 사전 학습된 resnext50_32x4d 모델 사용
model = models.resnext50_32x4d(pretrained=True)

# 모든 레이어 학습 가능하도록 설정 (고정 부분 제거)
# 이제 모든 파라미터가 학습에 참여함
for param in model.parameters():
    param.requires_grad = True

# 출력 레이어를 분류하려는 클래스 수에 맞게 수정
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.5),  # Dropout 추가
    torch.nn.Linear(model.fc.in_features, len(train_dataset.classes))
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 손실 함수 및 옵티마이저 설정
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.00005, momentum=0.9)

# 학습률 스케줄러 추가 (학습률 점진적 감소)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# 학습 과정 (TQDM으로 진행 상황 시각화)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # TQDM으로 학습 진행 표시
    train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, labels in train_loader_iter:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        train_loader_iter.set_postfix(loss=running_loss / len(train_loader))
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")
    
    # 학습률 스케줄러 업데이트
    scheduler.step()

# 모델 저장
torch.save(model.state_dict(), 'resnext_model.pth')

# 검증 평가
model.eval()
correct = 0
total = 0
with torch.no_grad():
    valid_loader_iter = tqdm(valid_loader, desc="Validating")  # 검증도 TQDM으로 진행 상황 시각화
    for images, labels in valid_loader_iter:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Validation Accuracy: {accuracy}%')


Epoch 1/10: 100%|█████████████████████████████████████████████████████████| 322/322 [02:18<00:00,  2.32it/s, loss=1.41]


Epoch 1/10, Loss: 1.4071759962887498


Epoch 2/10: 100%|██████████████████████████████████████████████████████████| 322/322 [02:19<00:00,  2.32it/s, loss=1.4]


Epoch 2/10, Loss: 1.3957509457694819


Epoch 3/10: 100%|█████████████████████████████████████████████████████████| 322/322 [02:19<00:00,  2.31it/s, loss=1.38]


...

Epoch 9/10, Loss: 1.351695092198271


Epoch 10/10: 100%|████████████████████████████████████████████████████████| 322/322 [02:18<00:00,  2.32it/s, loss=1.35]


Epoch 10/10, Loss: 1.345604768080741


Validating: 100%|██████████████████████████████████████████████████████████████████████| 41/41 [00:12<00:00,  3.30it/s]

Validation Accuracy: 36.993769470404985%
